# Embedding
  
Das Notebook dient dazu per:
  
- **CLIP**          CLIP -> 
- **AnyLoc**        DINOv2 -> Feature Aggregation -> Descriptor -> Retrival (Github: https://github.com/AnyLoc/Revisit-Anything.git)
- **EigenPlaces**   Backbone -> VPR-Descriptor -> Retrival (Github: https://github.com/gmberton/EigenPlaces.git)
- **MixVPR**        noch keine Idee (Mixed ansatz)
  
die Bilder in Vectorinformationen zu embedden


In [ ]:

import numpy as np
import pandas as pd

import sys
from pathlib import Path

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, paths
from src.device import pick_device

CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)

from src.run_guard import embedding_fingerprint, print_run_header, short_hash, write_fingerprint, validate_config
validate_config(CFG)

METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]


# Nur zum Ausprobieren auf einem Ausschnitt -- steht in der config.yaml,
# damit ein versehentlich gesetzter Wert im Diff sichtbar ist.
N_IMAGES = CFG["vpr"].get("max_images")
MAX_MISSING_FRAC = float(CFG["max_missing_images_frac"])
PROCESSED_DIR = PATHS.processed
IMAGE_PATH = PATHS.images
EMBEDDING_DIR = PATHS.embeddings
METHOD_DIR = PATHS.embedding_dir(METHOD)
METHOD_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"

# Erzeugt nur die Baseline. Die adaptierte Variante schreibt 05, und
# vpr.adapter waehlt in 06/07 aus, welche der beiden ausgewertet wird.
EMBEDDING_NAME = METHOD

metadata = pd.read_parquet(DATA_PATH_META)
embedding_metadata = metadata[metadata["split"].isin(["train", "database", "query"])].copy()
embedding_metadata = embedding_metadata.reset_index(drop=True)
embedding_path = METHOD_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = METHOD_DIR / f"{EMBEDDING_NAME}_metadata.parquet"


# Eine GPU macht den Encoder rund 20x schneller, noetig ist sie nicht.
DEVICE = pick_device()

# Grosse Modelle brauchen eine kleinere Batch als der Rest:
# vpr.<verfahren>.batch_size sticht vpr.embed_batch_size.
BATCH_SIZE = (CFG["vpr"].get(METHOD) or {}).get(
    "batch_size", CFG["vpr"]["embed_batch_size"]
)


# Wie viele Bilder eingebettet werden sollen
if N_IMAGES is not None:

    if N_IMAGES <= 0:
        raise ValueError("N_IMAGES muss None oder größer Null sein")
    
    if N_IMAGES > len(embedding_metadata):
        raise ValueError(
            f"N_IMAGES darf nicht größer als {len(embedding_metadata)} sein ist aber {N_IMAGES} "
        )
    embedding_metadata = embedding_metadata.iloc[:N_IMAGES].copy()


image_paths = [IMAGE_PATH / f"{i}.jpg" for i in embedding_metadata["image_id"]]

exists = np.array([p.exists() for p in image_paths])
anteil_fehlend = float((~exists).mean()) if len(exists) else 0.0

# Fehlende Bilder herauszufiltern ist kein harmloser Schoenheitsfehler: der
# Fingerabdruck entsteht danach aus den GEFILTERTEN Metadaten, passt also zu
# sich selbst, und keine spaetere Stufe merkt, dass ein Teil des Datensatzes
# fehlt. Die Warnung allein reicht nicht -- run.py fuehrt dieses Notebook
# ohne Konsole aus, prints landen nirgends. Deshalb eine harte Schranke.
if anteil_fehlend > MAX_MISSING_FRAC:
    raise RuntimeError(
        f"{(~exists).sum():,} von {len(exists):,} Bildern fehlen "
        f"({anteil_fehlend:.2%}) -- erlaubt sind {MAX_MISSING_FRAC:.2%} "
        "(config.yaml -> max_missing_images_frac).\n"
        f"Bildordner: {IMAGE_PATH}\n"
        "-> python run.py --from 03   (laedt nur die fehlenden nach)"
    )
if not exists.all():
    print(
        f"WARNUNG: {(~exists).sum():,} von {len(exists):,} Bildern fehlen "
        f"({anteil_fehlend:.3%}, unter der Schranke) -- werden uebersprungen."
    )
    embedding_metadata = embedding_metadata[exists].reset_index(drop=True)
    image_paths = [p for p, ok in zip(image_paths, exists) if ok]


print(f"Bilder:             {len(metadata):,}")
print(f"Bildordner:         {IMAGE_PATH}")
print(f"Embedding-Ordner:   {EMBEDDING_DIR}")
print(f"running on:         {DEVICE}")
print(f"batch size:         {BATCH_SIZE}")
print(f"Method:             {METHOD}")
print(f"MODEL_ID:           {MODEL_ID}")


# Modell laden
  
**MODEL_REVISION** = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268" for _reloading_ Modell



In [ ]:
from src.models.factory import build_embedder

embedder = build_embedder(METHOD, CFG, DEVICE, PROJECT_ROOT)

# AnyLoc passt seine PCA auf train-Bildern an. Das braucht Bildpfade und
# Metadaten, deshalb steht es hier und nicht in der Factory.
if METHOD == "anyloc" and CFG["vpr"]["anyloc"].get("pca_dim"):
    acfg = CFG["vpr"]["anyloc"]
    db_paths = [
        IMAGE_PATH / f"{i}.jpg"
        for i in embedding_metadata.loc[
            embedding_metadata["split"] == "train", "image_id"
        ]
    ]
    # Die PCA liegt neben den Embeddings: nur mit ihr landet ein neues
    # Bild (Demo) im selben Unterraum wie die Datenbank.
    embedder.fit_pca(db_paths, n_images=acfg["pca_fit_images"], batch_size=BATCH_SIZE,
                     seed=CFG["vpr"]["split_seed"],
                     save_path=METHOD_DIR / f"{EMBEDDING_NAME}_pca.npz")


EMBEDDING_DIM = embedder.embedding_dim

FINGERPRINT = embedding_fingerprint(CFG, METHOD, "none", embedding_metadata)
RUN_HASH = short_hash(FINGERPRINT)

print_run_header(CFG, "04_embeddings", device=DEVICE, run_hash=RUN_HASH)
print(f"Embedding dimension: {EMBEDDING_DIM}")


# Embedding Creation

In [ ]:
PARTIAL_PATH = METHOD_DIR / f"{EMBEDDING_NAME}_{RUN_HASH}.partial.npy"

embeddings = embedder.embed_images(
    image_paths,
    batch_size=BATCH_SIZE,
    # Run-Hash im Namen, damit kein Teil-Checkpoint aus einem Lauf mit
    # anderen Einstellungen fortgesetzt wird. Die Datei ist zugleich das
    # Ergebnis: embed_images schreibt in eine memmap, nicht in den RAM.
    checkpoint_path=PARTIAL_PATH,
    checkpoint_every=500,
)

assert len(embeddings) == len(embedding_metadata)
assert embeddings.shape[1] == EMBEDDING_DIM

# Blockweise pruefen. np.isfinite(x).all() legt ein Bool-Array in einem
# Viertel der Arraygroesse an, np.linalg.norm(x, axis=1) sogar eines in
# voller Groesse -- bei MegaLoc auf Jena (23,6 GB) hat genau das den Kernel
# gekillt. einsum ueber einen Block kommt ohne Temporaer in Arraygroesse aus.
BLOCK = 20_000
n_nicht_endlich = 0
n_min, n_max, n_summe = float("inf"), float("-inf"), 0.0
for _s in range(0, len(embeddings), BLOCK):
    _b = np.asarray(embeddings[_s : _s + BLOCK], dtype=np.float32)
    n_nicht_endlich += int((~np.isfinite(_b)).sum())
    _nr = np.sqrt(np.einsum("ij,ij->i", _b, _b))
    n_min = min(n_min, float(_nr.min()))
    n_max = max(n_max, float(_nr.max()))
    n_summe += float(_nr.sum())
    del _b, _nr

assert n_nicht_endlich == 0, f"{n_nicht_endlich:,} nicht endliche Werte in den Embeddings"

print(f"Shape:                 {embeddings.shape}")
print(f"Dtype:                 {embeddings.dtype}")
print(f"Im Speicher:           {'nein (memmap)' if isinstance(embeddings, np.memmap) else 'ja'}")
print(f"Normalized Minimum:    {n_min:.2f}")
print(f"Normalized Maximum:    {n_max:.2f}")
print(f"Normalized Mittelwert: {n_summe / len(embeddings):.2f}")

# Embedding Speichern


In [ ]:
# Die memmap aus 04 IST schon das Ergebnis -- sie liegt nur unter dem
# Checkpoint-Namen. Umbenennen statt np.save: kein zweites Mal 23,6 GB
# schreiben, kein zweites Mal lesen. Ohne Checkpoint (kleine Mengen) bleibt
# es beim gewoehnlichen Speichern.
if isinstance(embeddings, np.memmap):
    embeddings.flush()
    del embeddings              # mmap schliessen, bevor die Datei wandert
    PARTIAL_PATH.replace(embedding_path)
elif PARTIAL_PATH.exists() and not embedding_path.exists():
    PARTIAL_PATH.replace(embedding_path)        # Zelle erneut ausgefuehrt
else:
    np.save(embedding_path, embeddings)

# Der Fortschrittszaehler gehoert zum Checkpoint und hat jetzt ausgedient.
PARTIAL_PATH.with_name(PARTIAL_PATH.name + ".fortschritt.json").unlink(missing_ok=True)
PARTIAL_PATH.unlink(missing_ok=True)

embedding_metadata.to_parquet(metadata_path, index=False)

write_fingerprint(
    embedding_path,
    FINGERPRINT,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    n_images=len(embedding_metadata),
)

print(f"Embeddings gespeichert in:  {embedding_path}")
print(f"Metadaten gespeichert in:   {metadata_path}")
print(f"Fingerabdruck:              {RUN_HASH}")